# Post5G - Add 5G components to a running network

## Prerequisites
* You have successfully followed the [Simple Scenario](./simple.ipynb) tutorial
* You have basic Python knowledge
* You have a good understanding of the 5G architecture
* You have a correct understanding of OAI logs

## Experiment for this scenario

In the [simple scenario](./simple.ipynb) we deployed a 5G network from zero. In
this scenario, we will modify an existing 5G network. More specifically, we will
extend the 5G network deployed with the simple scenario by adding a second gNB,
called `gnb2` implemented with OAI. A third UE called `sim_ue3` will be added,
also implemented with OAI. It is connected to the gNB over a *rfsim* simulated
radio channel - we will investigate real radios in other scenarios.

The figure below illustrates this augmented network.

![Add component topology](./figures/add_component.png)

The UEs are configured as follows:

| UE | SUPI | S-NSSAI | DNN |
| --- | --- | --- | --- |
| `sim_ue1` | `001010000000100` | `internet` | `internet` |
| `sim_ue2` | `001010000000101` | `internet` | `internet` |
| `sim_ue3` | `001010000000110` | `internet` | `internet` |


## Recover metamodel and deployment model of an existing deployment

We assume that a 5G network is already deployed and running and has been
deployed with our orchestrator with the [simple scenario](./simple.ipynb). We
can then retrieve the metamodel and deployment model of this previous
deployment.

In our case, we extend the [simple scenario](./simple.ipynb) that saved its
two models in `state/metamodel.ues.yaml` and `state/deployment_model.ues.yaml`
files, respectively. Therefore, we do not need to define the models by
ourselves, instead we can recover them from the file and use them directly.

In [1]:
from metamodel.models.metamodel import MetaModel
from orchestrator.models.deployment import DeploymentModel

metamodel = MetaModel.from_yaml_file('state/metamodel.ues.yaml')
deployment_model = DeploymentModel.from_yaml_file('state/deployment_model.ues.yaml')

We can double check that the models correspond by printing them.

In [2]:
print (metamodel.to_yaml())
print (deployment_model.to_yaml())

ues:
  sim_ue1: {}
  sim_ue2:
    config:
      supi: '001010000000101'

params:
  k8s:
    master_node: node-core
    join_command: 'kubeadm join 10.123.34.62:6443 --token semj8t.4o7qofyxam9n2wgt
      --discovery-token-ca-cert-hash sha256:b33ff8cc9c5974a565ed73499b4507e68468d5421a84545a5939ae37f95e2963 '
    used_ips:
    - 10.244.103.207
    - 10.244.36.195
provisioners:
  my-provisionner:
    type: bi
    params:
      experiment: dsa_scenario2
      create_experiment: true
      experiment_duration: 8h
      node_duration: 8h
nodes:
  node-core:
    is_provisioned: true
    provisioner: my-provisionner
    ssh_info:
      hostname: 10.123.34.62
      user: dsaucezi
      jumphost:
        hostname: bastion1.theblueprintfactory.org
        user: proxy
        common_args: -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o
          ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o
          ControlPersist=7200s -o ControlPath=/tmp/cm-%C
      comm

## Adding a gNB

To add a gNB to the RAN we have to instantiate a `RANManagement` manager taking
the general metamodel and deployement model we just recovered.

In [3]:
from orchestrator.deployment import RANManagement
ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

We can then define the metamodel and deployment model of the gNB to add.

To simplify development we can define the metamodel and deployment model of the
component at once using a merged model. A merged model of a component is just
the concatenation of the metamodel and the deployment model in one single
dictionary. Internally, managers can distinguish parts that belong to the
metamodel and parts that belong to the deployment model and reconcile the merged
model with its own models.

In our case, the merged model is a simple as saying that the component is a gNB
and that its implementation is OAI. The other parameters will be derived
automatically by the manager. By default, if we don't specify a Radio Unit for a
gNB, fallback to the default case which is, in the case of OAI, a
radio channel simulated with `rfsim`. We show examples of more complex
configurations later in this tutorial and example with real radios in other
tutorials.

We can then add the component by calling the manager' `add_component` function
with the desired name for the gNB (i.e., `gnb2`) and its models. Then actually
deploy the gNB in the network by calling the `deploy` function of the manager
as usual.

In [4]:
import yaml
merged_model = yaml.safe_load(
"""
type: gnb
implementation: oai
""")

ran.add_component(name="gnb2", merged=merged_model)
await ran.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-gnb2' chart] *********

TASK [helm/prepare : Create Helm build directory for 'ueransim-gnb' chart] *****
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node-core]
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node-co

By default the OAI RAN components are deployed in the `oai-ran` namespace in the
cluster. We can verify that the new gNB has been deployed properly, by listing
all the pods of that namespace by running the `run_node_command` command on 
a `ProvisioningManagement` manager which models aer fed by the manager we just
used to deploy the new component (i.e., `ran`).

In [5]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(preceding=ran)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ran", "-o", "wide"],
    nodes=["node-ran"],
    become=True, quiet=True))["node-ran"])

NAME                        READY   STATUS    RESTARTS   AGE   IP               NODE       NOMINATED NODE   READINESS GATES
oai-gnb2-6c6dcf89c5-flq5s   1/1     Running   0          3s    10.244.113.144   node-ran   <none>           <none>


To investigate if the gNB is up and running we can check the logs of the gNB by
calling the `get_logs` function on the manager. As we can see, the
`amf_ip_address` correspond to the address of the AMF of the core we deployed
with the [simple tutorial](./simple.ipynb) we can also see that the NGAP
procedure succeeded and that the gNB is on the air

In [6]:
print(await ran.get_logs(components=["gnb2"], quiet=True))

/proc/sys/kernel/core_pattern=|/usr/share/apport/apport -p%p -s%s -c%c -d%d -P%P -u%u -g%g -F%F -- %E
No configuration file /opt/oai-gnb/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

Active_gNBs:
  - gnb2
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  ##### Identification parameters:
  - gNB_ID: 0x1

    gNB_name: gnb2

    tracking_area_code: 1

    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1

    ##### Physical parameters:
    min_rxtxtime: 6

    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 0
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 621312
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 620040
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15,

## Adding a UE

As the RAN is up and running we can add the new UE. We follow the same approach
as for adding the gNB. As it is a UE component, we instantiate a `UEManagement`
manager.

In [7]:
from orchestrator.deployment import UEManagement
ues = UEManagement(preceding=ran)


In the model of the UE, we specify the implementation we desire to use and its
*Subscription Permanent Identifier* (SUPI). In addition, we define to which
gNBs the UE must connect (i.e., `gnb2`).

We can then add the UE as a new component that we name `sim_ue3`.

In [8]:
merged_model = yaml.safe_load(
"""
type: containerized
implementation: oai
config:
    supi: '001010000000110'
    gnbs:
    - gnb2
"""
)

ues.add_component(name="sim_ue3", merged=merged_model)


However, we cannot deploy the UE just yet as otherwise it would be rejected by
the core. Indeed, the UE's subscription must first be added to the core. To do
that we call the `register_ues` function for this UE on the `CoreManagement`
manager.

In [9]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(preceding=ues)
await core.register_ues(ues=["sim_ue3"])


PLAY [node-core] ***************************************************************

TASK [database/subscriber/add/open5gs : Add UE 'sim_ue3' to Open5GS database] ***
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
[WARNING]: Deprecation warnings can be disabled by setting `deprecation_warnings=False` in ansible.cfg.
[DEPRECATION WARNING]: Importing 'to_native' from 'ansible.module_utils._text' is deprecated. This feature will be removed from ansible-core version 2.24. Use ansible.module_utils.common.text.converters instead.
changed: [node-core]

PLAY RECAP *********************************************************************
node-core                  : ok=1    changed=1    unreachable=0    failed=0    skipped=0    re


Once the UE is registered in the core, we can deploy it and it will be able to
properly register to the network.

**NOTE:** remember the declarative functonal paradigm followed by the
orchestrator library, so do not forget to always instantiate managers with the 
metamodel and deployment model of the latest deployment stage.

In [10]:
ues = UEManagement(preceding=core)
await ues.deploy(components=["sim_ue3"])


PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue3' chart] ******
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node-core]

TASK [helm/prepare : Copy local 'oai-containerized' Helm chart directory for 'oai-sim-ue3'] ***
ok: [node-core]

TASK [helm/prepare : Render 'oai-sim-ue3' Helm Chart.yaml file from Jinja template] ***
ok: [node-core]

TASK [helm/prepare : Render templated Helm files for 'oai-sim-ue3'] ************
ok: [node-core] => (item=values.yaml)
changed: [node-core] => (item=config.yaml)

PLAY RECAP *********************************************************************
node-core                  : ok=4  

OAI UEs are deployed in the `oai-ue` namespace and by listing the pods in that
namespace we can see that the UE has been deployed properly

**NOTE1:** *Why do we use the `prov` manager instance instead of new instance of
`ProvisioningManagement` with the latest metamodel and deployment model?*
Managers only use the section of the metamodel and deployment model that they
need. In our case the changes in the models that occured after the instantiation
of `prov` would have no impact on the `ProvisioningManagement`, we can thus
safely use this particular instance.

**NOTE2:** *Why can we run the `kubectl` command on the `node-ran` machine while
UEs are deployed on `node-ues`?* Our kubernetes manager creates only one
kubernetes cluster that is composed of all the nodes. It installs the
`kubeconfig` file on all nodes composing the cluster meaning that kubernetes
actions can be launched from any machine of the cluster.

In [11]:
print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ue", "-o", "wide"],
    nodes=["node-ran"],
    become=True, quiet=True))["node-ran"])

NAME                          READY   STATUS    RESTARTS   AGE   IP             NODE       NOMINATED NODE   READINESS GATES
oai-sim-ue3-548cc87c8-fpvdc   1/1     Running   0          3s    10.244.198.7   node-ues   <none>           <none>


By showing the logs of the UE we can also verify that it has started correctly
and that it received an IP address on its radio interface named `oaitun_ue1`.

In [12]:
print(await ues.get_logs(components=["sim_ue3"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-c8b78200bf7392b842f7e5b8485837531acaa3e4 Ab

As we did with the two other UEs, we can ping `8.8.8.8` from this UE over its
radio interface showing that our 5G network is fully functional.

In [13]:
command = ["ping", "-I", "oaitun_ue1", "-c", 10, "-D", "8.8.8.8"]
pings = await ues.run_command(command=command, components=["sim_ue3"], quiet=True)
print (pings["sim_ue3"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.4 oaitun_ue1: 56(84) bytes of data.
[1791377696.997231] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=23.0 ms
[1791377697.998619] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=23.3 ms
[1791377698.999768] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=23.1 ms
[1791377699.998497] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=20.6 ms
[1791377700.994591] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=16.0 ms
[1791377702.003710] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=24.8 ms
[1791377703.004868] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=24.8 ms
[1791377704.000307] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=18.4 ms
[1791377705.007397] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=24.0 ms
[1791377706.010209] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=25.6 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9010ms
rtt min/avg/max/mdev = 15.978/22.370/25.641/2.955 ms



**NOTE:** We notice that the delay is much higher than when the ping was
performend on the UERANSIM UEs. The reason is because UERAN SIM does not
implement the radio while with OAI gNBs and rfsim, the radio is fully
implemented and simulated, hence the delay that are in adequation with the
resource block and TDD patterns.

To further verify the correctness of our 5G network, we can try to ping another
UE registered in the network from our freshly added UE. Let's ping `sim_ue1`.

We can retrieve the IP address of `sim_ue1` by running the `ip add show` command
on it (or manualy by looking a the the UEs logs that we extracted in the
[simple tutorial](./simple.ipynb))

In [14]:
command = ["bash", "-c", "'ip -4 addr show uesimtun0 | grep -oP \"(?<=inet )\\d+(\\.\\d+){3}\"'"]
sim_ue1_ip = (await ues.run_command(command=command, components=["sim_ue1"], quiet=True))["sim_ue1"].rstrip()
print (f"IP address of sim_ue1 {sim_ue1_ip}")

IP address of sim_ue1 11.1.1.2


In [15]:
command = ["ping", "-I", "oaitun_ue1", "-c", 10, "-D", sim_ue1_ip]
pings = await ues.run_command(command=command, components=["sim_ue3"], quiet=True)
print (pings["sim_ue3"])

PING 11.1.1.2 (11.1.1.2) from 11.1.1.4 oaitun_ue1: 56(84) bytes of data.
[1791377711.171634] 64 bytes from 11.1.1.2: icmp_seq=1 ttl=64 time=21.8 ms
[1791377712.173653] 64 bytes from 11.1.1.2: icmp_seq=2 ttl=64 time=22.0 ms
[1791377713.174083] 64 bytes from 11.1.1.2: icmp_seq=3 ttl=64 time=22.0 ms
[1791377714.174922] 64 bytes from 11.1.1.2: icmp_seq=4 ttl=64 time=22.2 ms
[1791377715.176235] 64 bytes from 11.1.1.2: icmp_seq=5 ttl=64 time=22.5 ms
[1791377716.179914] 64 bytes from 11.1.1.2: icmp_seq=6 ttl=64 time=24.5 ms
[1791377717.178659] 64 bytes from 11.1.1.2: icmp_seq=7 ttl=64 time=21.7 ms
[1791377718.176631] 64 bytes from 11.1.1.2: icmp_seq=8 ttl=64 time=18.5 ms
[1791377719.180478] 64 bytes from 11.1.1.2: icmp_seq=9 ttl=64 time=20.8 ms
[1791377720.180828] 64 bytes from 11.1.1.2: icmp_seq=10 ttl=64 time=19.3 ms

--- 11.1.1.2 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9012ms
rtt min/avg/max/mdev = 18.494/21.517/24.503/1.592 ms



Finally, we can archive the metamodel and deployment model in file for keeping
track of the experiment models.

In [16]:
from pathlib import Path
DIR="state/add_components"
Path(DIR).mkdir(parents=True, exist_ok=True)

ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")